# Task 3: Data Preprocessing

## 1. Project Overview

This notebook focuses on preparing a news article dataset
containing approximately 100,000 records for further analysis.

The preprocessing workflow will begin by inspecting the dataset
and understanding its structure, data types, and data quality.

## 2. Objectives

- Load the original news article dataset.
- Inspect rows, columns, and data types.
- Identify missing values and duplicate records.
- Clean text and handle data quality issues where required.
- Validate and export the processed dataset.

## 3. Dataset

**Input file:** `articles_100k_dataset.csv`

**Source:** News article dataset prepared in Task 1.

**Approach:** Preserve the original dataset and save the
preprocessed output separately.

## 4. Workflow

1. Dataset loading
2. Initial data inspection
3. Missing-value analysis
4. Duplicate analysis
5. Data cleaning and validation
6. Export of the processed dataset

## 5. Load the Dataset

In this section, the original CSV dataset is loaded into a Pandas DataFrame. The initial inspection will help us understand the dataset's size, column names, data types, and sample records before applying any preprocessing operations.

In [2]:
import pandas as pd
from pathlib import Path

# Define the input dataset path
DATASET_PATH = "articles_100k_dataset.csv"

# Load the original dataset
df = pd.read_csv(DATASET_PATH, low_memory=False)

# Display dataset information
print("Dataset loaded successfully!")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types and Non-Null Counts:")
df.info()

print("\nFirst 5 Records:")
display(df.head())

Dataset loaded successfully!
Number of rows: 100,000
Number of columns: 9

Column Names:
['article_id', 'source', 'title', 'description', 'author', 'url', 'published_at', 'timezone', 'scraped_at']

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   article_id    100000 non-null  object
 1   source        100000 non-null  object
 2   title         100000 non-null  object
 3   description   99877 non-null   object
 4   author        41539 non-null   object
 5   url           100000 non-null  object
 6   published_at  100000 non-null  object
 7   timezone      100000 non-null  object
 8   scraped_at    100000 non-null  object
dtypes: object(9)
memory usage: 6.9+ MB

First 5 Records:


,article_id,source,title,description,author,url,published_at,timezone,scraped_at
0,431f5566a31b29b4,BBC News,"Almost 1,000 arrested as clashes follow anothe...",Hundreds of thousands took part in demonstrati...,NaN,https://www.bbc.co.uk/news/articles/c95ynd587y...,2026-10-25T04:16:52+00:00,UTC,2026-10-25T06:41:03.641673+00:00
1,ff00da6ff48d059a,BBC News,Israel to close West Bank crossings on electio...,The Israeli Defense Forces announces restricti...,NaN,https://www.bbc.co.uk/news/articles/c6y93ylpl5...,2026-10-25T05:15:52+00:00,UTC,2026-10-25T06:41:07.354368+00:00
2,ee273e8d6d2effa2,BBC News,Voters in Brazil to choose between Lula and Bo...,"Flávio Bolsonaro, the son of Brazil's imprison...",NaN,https://www.bbc.co.uk/news/articles/cxp8g21j46...,2026-10-24T23:50:02+00:00,UTC,2026-10-25T06:41:11.347318+00:00
3,af154c8b7028fa6b,BBC News,Ukraine says Starlink connection 'now stable' ...,The slowed Starlink connections had affected t...,NaN,https://www.bbc.co.uk/news/articles/cwkg5md14j...,2026-10-24T12:12:40+00:00,UTC,2026-10-25T06:41:15.790533+00:00
4,35c67fdb10b3910b,BBC News,Gunmen kill at least 10 villagers and loot sho...,"Police are searching for other casualties, wit...",NaN,https://www.bbc.co.uk/news/articles/cq1kdlm1ze...,2026-10-25T03:20:16+00:00,UTC,2026-10-25T06:41:20.034129+00:00


## 6. Missing Values Analysis

In [3]:

# Calculate missing values for each column
missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage (%)": (df.isnull().sum() / len(df)) * 100,
    "Available Values": df.notnull().sum()
})

# Sort columns by missing values in descending order
missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

# Display the missing values report
print("Missing Values Report")
display(missing_summary.round(2))

# Calculate total missing cells in the dataset
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]

print(f"Total missing cells: {total_missing:,}")
print(f"Total dataset cells: {total_cells:,}")
print(f"Overall missing percentage: {(total_missing / total_cells) * 100:.2f}%")


Missing Values Report


,Missing Values,Missing Percentage (%),Available Values
author,58461,58.46,41539
description,123,0.12,99877
article_id,0,0.00,100000
title,0,0.00,100000
source,0,0.00,100000
url,0,0.00,100000
published_at,0,0.00,100000
timezone,0,0.00,100000
scraped_at,0,0.00,100000


Total missing cells: 58,584
Total dataset cells: 900,000
Overall missing percentage: 6.51%


## 7. Duplicate Records Analysis

In [4]:

# Check completely duplicated rows
duplicate_rows = df.duplicated().sum()

# Check repeated article IDs
duplicate_article_ids = df["article_id"].duplicated().sum()

# Check repeated URLs
duplicate_urls = df["url"].duplicated().sum()

# Display duplicate analysis
print("Duplicate Records Analysis")
print("-" * 40)
print(f"Completely duplicated rows: {duplicate_rows:,}")
print(f"Duplicate article IDs: {duplicate_article_ids:,}")
print(f"Duplicate URLs: {duplicate_urls:,}")

# Display sample repeated URLs, if any
if duplicate_urls > 0:
    print("\nSample records with repeated URLs:")
    display(
        df[df["url"].duplicated(keep=False)]
        [["article_id", "source", "title", "url"]]
        .head(10)
    )
else:
    print("\nNo repeated URLs found.")


Duplicate Records Analysis
----------------------------------------
Completely duplicated rows: 0
Duplicate article IDs: 0
Duplicate URLs: 0

No repeated URLs found.


## 8. Data Types and Date-Time Validation

In [5]:

# Identify date-time columns
datetime_columns = ["published_at", "scraped_at"]

print("Date-Time Validation")
print("-" * 40)

for column in datetime_columns:
    # Count values that cannot be parsed as valid timestamps
    parsed_values = pd.to_datetime(df[column], errors="coerce", utc=True)
    invalid_count = parsed_values.isna().sum()

    print(f"\nColumn: {column}")
    print(f"Original data type: {df[column].dtype}")
    print(f"Invalid or missing timestamps: {invalid_count:,}")
    print("Sample values:")
    display(df[column].head(3))

# Check timezone values
print("\nTimezone Distribution")
display(df["timezone"].value_counts(dropna=False).head(10))


Date-Time Validation
----------------------------------------

Column: published_at
Original data type: object
Invalid or missing timestamps: 31,214
Sample values:


0    2026-10-25T04:16:52+00:00
1    2026-10-25T05:15:52+00:00
2    2026-10-24T23:50:02+00:00
Name: published_at, dtype: object


Column: scraped_at
Original data type: object
Invalid or missing timestamps: 849
Sample values:


0    2026-10-25T06:41:03.641673+00:00
1    2026-10-25T06:41:07.354368+00:00
2    2026-10-25T06:41:11.347318+00:00
Name: scraped_at, dtype: object


Timezone Distribution


timezone
UTC          80457
unknown      13332
UTC+02:00     4425
UTC-04:00     1715
UTC+01:00       47
UTC-05:00       13
Unknown         11
Name: count, dtype: int64

## 9. Investigation of Invalid Timestamps

In [6]:

# Identify timestamps that could not be parsed
published_parsed = pd.to_datetime(
    df["published_at"], errors="coerce", utc=True
)

scraped_parsed = pd.to_datetime(
    df["scraped_at"], errors="coerce", utc=True
)

# Inspect invalid published_at values
invalid_published = df.loc[
    published_parsed.isna(), ["source", "title", "published_at"]
]

print("Invalid published_at records:")
print(f"Total: {len(invalid_published):,}")
display(invalid_published.head(15))

# Inspect invalid scraped_at values
invalid_scraped = df.loc[
    scraped_parsed.isna(), ["source", "title", "scraped_at"]
]

print("\nInvalid scraped_at records:")
print(f"Total: {len(invalid_scraped):,}")
display(invalid_scraped.head(15))

# Check whether invalid timestamps are empty or contain text
print("\nMissing/blank published_at values:")
print(
    df["published_at"].isna().sum()
    + df["published_at"].astype("string").str.strip().eq("").sum()
)

print("\nMissing/blank scraped_at values:")
print(
    df["scraped_at"].isna().sum()
    + df["scraped_at"].astype("string").str.strip().eq("").sum()
)


Invalid published_at records:
Total: 31,214


,source,title,published_at
300,The Guardian,Pakistan v England: tri-series one day interna...,2026-10-25T07:59:38.000Z
301,BBC,Test - BBC News Afrique,2014-09-24T14:32:16.000Z
302,Euronews,Serbia goes to polls to choose between Vučić a...,2026-10-25 08:24:52 +01:00
304,Sky News,Make Tracks - 4th Dec 2024,2024-12-04
305,TechCrunch,stuff | TechCrunch,2026-10-25
306,Axios,Washington's message to AI execs: please shut up,2026-10-24T16:32:03.638000Z
308,NPR,"Peltola, Democratic Party move to intervene in...",2026-10-24T21:39:13.133
309,The Independent,Saudi Arabia reopens airport as attacks by Hou...,2026-10-25T07:30:49.000Z
310,Euronews,Should Europe ban meat billboards? Climate psy...,2026-10-25 08:02:08 +01:00
311,TechCrunch,media | TechCrunch,2026-10-25



Invalid scraped_at records:
Total: 849


,source,title,scraped_at
39527,BBC News,US citizen Stephen Hubbard freed by Russia as ...,2026-10-21T15:26:27+00:00
39528,BBC News,Macron accuses foreign leaders of exploiting F...,2026-10-21T15:26:28+00:00
39529,BBC News,Helen Haines: Australian MP loses both legs af...,2026-10-21T15:26:29+00:00
39530,BBC News,Elon Musk confirms he is considering merging T...,2026-10-21T15:26:30+00:00
39531,BBC News,Pakistan arrests man wanted by US over $126m p...,2026-10-21T15:26:31+00:00
39532,BBC News,The Gambia to pay Yahya Jammeh victims from th...,2026-10-21T15:26:32+00:00
39533,BBC News,Brian Stack: Family of prison officer killed b...,2026-10-21T15:26:33+00:00
39534,BBC News,Nigeria confirms Ebola case linked to East Africa,2026-10-21T15:26:33+00:00
39535,BBC News,Sudan army drones hit mourners and fuel market...,2026-10-21T15:26:34+00:00
39536,BBC News,Trump and Zelensky at loggerheads despite Oslo...,2026-10-21T15:26:35+00:00



Missing/blank published_at values:
0

Missing/blank scraped_at values:
0


## 10. Date-Time Standardization

In [7]:

# Create temporary standardized datetime values
published_standardized = pd.to_datetime(
    df["published_at"],
    format="mixed",
    errors="coerce",
    utc=True
)

scraped_standardized = pd.to_datetime(
    df["scraped_at"],
    format="mixed",
    errors="coerce",
    utc=True
)

# Compare parsing results
datetime_report = pd.DataFrame({
    "Column": ["published_at", "scraped_at"],
    "Original Invalid Count": [
        pd.to_datetime(df["published_at"], errors="coerce", utc=True).isna().sum(),
        pd.to_datetime(df["scraped_at"], errors="coerce", utc=True).isna().sum()
    ],
    "Standardized Invalid Count": [
        published_standardized.isna().sum(),
        scraped_standardized.isna().sum()
    ],
    "Successfully Parsed": [
        published_standardized.notna().sum(),
        scraped_standardized.notna().sum()
    ]
})

display(datetime_report)

# Display examples that remain unparseable
print("Unparseable published_at examples:")
display(
    df.loc[published_standardized.isna(), ["source", "title", "published_at"]]
      .head(10)
)

print("Unparseable scraped_at examples:")
display(
    df.loc[scraped_standardized.isna(), ["source", "title", "scraped_at"]]
      .head(10)
)


,Column,Original Invalid Count,Standardized Invalid Count,Successfully Parsed
0,published_at,31214,0,100000
1,scraped_at,849,0,100000


Unparseable published_at examples:


,source,title,published_at


Unparseable scraped_at examples:


,source,title,scraped_at


## 11. Validation of Standardized Timestamps

In [8]:

# Create a temporary validation DataFrame
datetime_validation = pd.DataFrame({
    "original_published_at": df["published_at"].head(5),
    "standardized_published_at": published_standardized.head(5),
    "original_scraped_at": df["scraped_at"].head(5),
    "standardized_scraped_at": scraped_standardized.head(5)
})

print("Original and Standardized Timestamp Comparison:")
display(datetime_validation)

print("\nStandardized Data Types:")
print("published_at:", published_standardized.dtype)
print("scraped_at:", scraped_standardized.dtype)

print("\nValidation Checks:")
print(
    "All published_at values parsed:",
    published_standardized.notna().all()
)
print(
    "All scraped_at values parsed:",
    scraped_standardized.notna().all()
)


Original and Standardized Timestamp Comparison:


,original_published_at,standardized_published_at,original_scraped_at,standardized_scraped_at
0,2026-10-25T04:16:52+00:00,2026-10-25 04:16:52+00:00,2026-10-25T06:41:03.641673+00:00,2026-10-25 06:41:03.641673+00:00
1,2026-10-25T05:15:52+00:00,2026-10-25 05:15:52+00:00,2026-10-25T06:41:07.354368+00:00,2026-10-25 06:41:07.354368+00:00
2,2026-10-24T23:50:02+00:00,2026-10-24 23:50:02+00:00,2026-10-25T06:41:11.347318+00:00,2026-10-25 06:41:11.347318+00:00
3,2026-10-24T12:12:40+00:00,2026-10-24 12:12:40+00:00,2026-10-25T06:41:15.790533+00:00,2026-10-25 06:41:15.790533+00:00
4,2026-10-25T03:20:16+00:00,2026-10-25 03:20:16+00:00,2026-10-25T06:41:20.034129+00:00,2026-10-25 06:41:20.034129+00:00



Standardized Data Types:
published_at: datetime64[ns, UTC]
scraped_at: datetime64[ns, UTC]

Validation Checks:
All published_at values parsed: True
All scraped_at values parsed: True


## 12. Timezone Standardization

In [9]:

# Inspect original timezone distribution
print("Original Timezone Distribution:")
display(df["timezone"].value_counts(dropna=False))

# Create a standardized copy of the timezone column
timezone_standardized = (
    df["timezone"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Treat blank values as missing
timezone_standardized = timezone_standardized.replace("", pd.NA)

# Standardize unknown labels
timezone_standardized = timezone_standardized.replace(
    ["UNKNOWN", "N/A", "NA", "NONE"],
    "UNKNOWN"
)

print("\nStandardized Timezone Distribution:")
display(timezone_standardized.value_counts(dropna=False))

# Count values changed during standardization
changed_values = (
    df["timezone"].astype("string").fillna("<MISSING>")
    != timezone_standardized.fillna("<MISSING>")
).sum()

print(f"\nRecords affected by standardization: {changed_values:,}")
print(f"Missing timezone values after standardization: {timezone_standardized.isna().sum():,}")


Original Timezone Distribution:


timezone
UTC          80457
unknown      13332
UTC+02:00     4425
UTC-04:00     1715
UTC+01:00       47
UTC-05:00       13
Unknown         11
Name: count, dtype: int64


Standardized Timezone Distribution:


timezone
UTC          80457
UNKNOWN      13343
UTC+02:00     4425
UTC-04:00     1715
UTC+01:00       47
UTC-05:00       13
Name: count, dtype: Int64


Records affected by standardization: 13,343
Missing timezone values after standardization: 0


## 13. Text Data Quality Inspection

In [10]:

# Text columns to inspect
text_columns = ["source", "title", "description", "author", "url"]

print("Text Data Quality Report")
print("-" * 50)

for column in text_columns:
    values = df[column].astype("string")

    missing_count = df[column].isna().sum()
    blank_count = values.str.strip().eq("").fillna(False).sum()
    whitespace_count = (
        values.notna() & values.ne(values.str.strip())
    ).sum()

    print(f"\nColumn: {column}")
    print(f"Missing values: {missing_count:,}")
    print(f"Blank/empty strings: {blank_count:,}")
    print(f"Values with leading/trailing whitespace: {whitespace_count:,}")

# Inspect unusually short titles
title_lengths = df["title"].astype("string").str.strip().str.len()

print("\nTitles with fewer than 10 characters:")
short_titles = df.loc[
    title_lengths.lt(10).fillna(False),
    ["source", "title", "url"]
]

print(f"Count: {len(short_titles):,}")
display(short_titles.head(15))

# Inspect short descriptions
description_lengths = df["description"].astype("string").str.strip().str.len()

print("\nNon-missing descriptions with fewer than 20 characters:")
short_descriptions = df.loc[
    df["description"].notna()
    & description_lengths.lt(20).fillna(False),
    ["source", "title", "description"]
]

print(f"Count: {len(short_descriptions):,}")
display(short_descriptions.head(10))


Text Data Quality Report
--------------------------------------------------

Column: source
Missing values: 0
Blank/empty strings: 0
Values with leading/trailing whitespace: 0

Column: title
Missing values: 0
Blank/empty strings: 0
Values with leading/trailing whitespace: 0

Column: description
Missing values: 123
Blank/empty strings: 0
Values with leading/trailing whitespace: 0

Column: author
Missing values: 58,461
Blank/empty strings: 0
Values with leading/trailing whitespace: 0

Column: url
Missing values: 0
Blank/empty strings: 0
Values with leading/trailing whitespace: 0

Titles with fewer than 10 characters:
Count: 27


,source,title,url
3326,The Verge,Skydrop.,https://www.theverge.com/entertainment/1012471...
4692,NPR,Dabin Han,https://www.npr.org/people/g-s1-148552/dabin-han
16396,NPR,Friko,https://www.npr.org/artists/g-s1-145555/friko
21017,NPR,Elim Chan,https://www.npr.org/artists/g-s1-144310/elim-chan
23457,Axios,News quiz,https://www.axios.com/local/tampa-bay/2025/04/...
27104,The Verge,All rise.,https://www.theverge.com/entertainment/969954/...
27169,The Verge,Splat!,https://www.theverge.com/entertainment/969898/...
30541,NPR,Quizzes,https://www.npr.org/series/g-s1-142037/quizzes
31659,The Verge,Waaah!,https://www.theverge.com/entertainment/962712/...
31763,The Verge,“Ah.”,https://www.theverge.com/games/962549/ah



Non-missing descriptions with fewer than 20 characters:
Count: 241


,source,title,description
301,BBC,Test - BBC News Afrique,Test
411,The Verge,Jack Dorsey’s Block made an AI model that talk...,"Hello, Dessa?"
1007,The Verge,Fall 2026 earnings watch,Tune into Q3.
1184,The Verge,Someone hacked the Switch 2 by using an old ga...,The door is open.
2010,The Verge,Apple Music has a new logo,Boom.
2227,The Verge,Snap is paying Robert Downey Jr. to do the Iro...,He is Specs man.
2252,The Verge,Logitech’s refurbished G Pro X2 Superstrike is...,A buzzworthy deal.
2958,The Verge,"Wired earbuds are having a moment, and Bose is...",Wired? Inspired.
2995,The Verge,All the news about the iPhone Duo,Time to fold.
3169,The Verge,iPhone Duo orders are already delayed into 2027,Duo delay.


## 14. Text Cleaning and Missing-Value Handling

In [11]:

# Create a separate copy for preprocessing
df_clean = df.copy()

# Clean whitespace in text columns without changing the original DataFrame
text_columns = [
    "source", "title", "description", "author", "url", "timezone"
]

for column in text_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

# Standardize timezone labels
df_clean["timezone"] = (
    df_clean["timezone"]
    .str.upper()
    .replace(["", "N/A", "NA", "NONE"], pd.NA)
    .fillna("UNKNOWN")
)

# Standardize unknown labels
df_clean["timezone"] = df_clean["timezone"].replace("UNKNOWN", "UNKNOWN")

# Convert empty text values to missing values where appropriate
for column in ["description", "author"]:
    df_clean[column] = df_clean[column].replace("", pd.NA)

# Apply the validated datetime conversions
df_clean["published_at"] = published_standardized
df_clean["scraped_at"] = scraped_standardized

print("Text cleaning completed.")
print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Original columns: {df.shape[1]}")
print(f"Cleaned columns: {df_clean.shape[1]}")

print("\nMissing Values After Cleaning:")
display(df_clean.isna().sum().to_frame("Missing Values"))

print("\nData Types After Cleaning:")
display(df_clean.dtypes.to_frame("Data Type"))

print("\nCleaned Dataset Preview:")
display(df_clean.head())


Text cleaning completed.
Original rows: 100,000
Cleaned rows: 100,000
Original columns: 9
Cleaned columns: 9

Missing Values After Cleaning:


,Missing Values
article_id,0
source,0
title,0
description,123
author,58461
url,0
published_at,0
timezone,0
scraped_at,0



Data Types After Cleaning:


,Data Type
article_id,object
source,string[python]
title,string[python]
description,string[python]
author,string[python]
url,string[python]
published_at,"datetime64[ns, UTC]"
timezone,string[python]
scraped_at,"datetime64[ns, UTC]"



Cleaned Dataset Preview:


,article_id,source,title,description,author,url,published_at,timezone,scraped_at
0,431f5566a31b29b4,BBC News,"Almost 1,000 arrested as clashes follow anothe...",Hundreds of thousands took part in demonstrati...,<NA>,https://www.bbc.co.uk/news/articles/c95ynd587y...,2026-10-25 04:16:52+00:00,UTC,2026-10-25 06:41:03.641673+00:00
1,ff00da6ff48d059a,BBC News,Israel to close West Bank crossings on electio...,The Israeli Defense Forces announces restricti...,<NA>,https://www.bbc.co.uk/news/articles/c6y93ylpl5...,2026-10-25 05:15:52+00:00,UTC,2026-10-25 06:41:07.354368+00:00
2,ee273e8d6d2effa2,BBC News,Voters in Brazil to choose between Lula and Bo...,"Flávio Bolsonaro, the son of Brazil's imprison...",<NA>,https://www.bbc.co.uk/news/articles/cxp8g21j46...,2026-10-24 23:50:02+00:00,UTC,2026-10-25 06:41:11.347318+00:00
3,af154c8b7028fa6b,BBC News,Ukraine says Starlink connection 'now stable' ...,The slowed Starlink connections had affected t...,<NA>,https://www.bbc.co.uk/news/articles/cwkg5md14j...,2026-10-24 12:12:40+00:00,UTC,2026-10-25 06:41:15.790533+00:00
4,35c67fdb10b3910b,BBC News,Gunmen kill at least 10 villagers and loot sho...,"Police are searching for other casualties, wit...",<NA>,https://www.bbc.co.uk/news/articles/cq1kdlm1ze...,2026-10-25 03:20:16+00:00,UTC,2026-10-25 06:41:20.034129+00:00


## 15. Final Data Quality Validation

In [12]:

# Final validation report
validation_report = {
    "Total Rows": len(df_clean),
    "Total Columns": df_clean.shape[1],
    "Duplicate Rows": int(df_clean.duplicated().sum()),
    "Duplicate Article IDs": int(df_clean["article_id"].duplicated().sum()),
    "Duplicate URLs": int(df_clean["url"].duplicated().sum()),
    "Invalid published_at": int(df_clean["published_at"].isna().sum()),
    "Invalid scraped_at": int(df_clean["scraped_at"].isna().sum()),
    "Missing Authors": int(df_clean["author"].isna().sum()),
    "Missing Descriptions": int(df_clean["description"].isna().sum()),
    "Missing Timezones": int(df_clean["timezone"].isna().sum())
}

print("FINAL DATA QUALITY REPORT")
print("=" * 40)
display(pd.DataFrame(
    list(validation_report.items()),
    columns=["Validation Check", "Result"]
))

# Verify required fields
required_fields = ["article_id", "source", "title", "url"]

print("\nRequired Field Validation")
for column in required_fields:
    print(f"{column}: {df_clean[column].notna().all()}")

# Check timestamp order
dates_out_of_order = (
    df_clean["scraped_at"] < df_clean["published_at"]
).sum()

print(f"\nRecords scraped before publication: {dates_out_of_order:,}")

# Confirm schema
print("\nFinal Column Order:")
print(df_clean.columns.tolist())


FINAL DATA QUALITY REPORT


,Validation Check,Result
0,Total Rows,100000
1,Total Columns,9
2,Duplicate Rows,0
3,Duplicate Article IDs,0
4,Duplicate URLs,0
5,Invalid published_at,0
6,Invalid scraped_at,0
7,Missing Authors,58461
8,Missing Descriptions,123
9,Missing Timezones,0



Required Field Validation
article_id: True
source: True
title: True
url: True

Records scraped before publication: 0

Final Column Order:
['article_id', 'source', 'title', 'description', 'author', 'url', 'published_at', 'timezone', 'scraped_at']


## 16. Export the Preprocessed Dataset

In [14]:

from pathlib import Path

# Create an output directory
OUTPUT_DIR = Path("preprocessed_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define output file paths
OUTPUT_FILE = OUTPUT_DIR / "preprocessed_articles_100k.csv"
REPORT_FILE = OUTPUT_DIR / "preprocessing_summary.csv"

# Export the cleaned dataset
df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# Create a preprocessing summary report
summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Columns",
        "Duplicate Rows",
        "Duplicate Article IDs",
        "Duplicate URLs",
        "Missing Authors",
        "Missing Descriptions",
        "Invalid Publication Timestamps",
        "Invalid Scraping Timestamps",
        "Missing Timezones"
    ],
    "Value": [
        len(df_clean),
        df_clean.shape[1],
        df_clean.duplicated().sum(),
        df_clean["article_id"].duplicated().sum(),
        df_clean["url"].duplicated().sum(),
        df_clean["author"].isna().sum(),
        df_clean["description"].isna().sum(),
        df_clean["published_at"].isna().sum(),
        df_clean["scraped_at"].isna().sum(),
        df_clean["timezone"].isna().sum()
    ]
})

summary.to_csv(REPORT_FILE, index=False, encoding="utf-8-sig")

# Verify the exported CSV
exported_df = pd.read_csv(OUTPUT_FILE, low_memory=False)

print("Export completed successfully!")
print(f"\nDataset file: {OUTPUT_FILE}")
print(f"Summary report: {REPORT_FILE}")
print(f"\nExported rows: {len(exported_df):,}")
print(f"Exported columns: {len(exported_df.columns)}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024 * 1024):.2f} MB")

print("\nExport verification:")
print("Row count preserved:", len(exported_df) == len(df_clean))
print("Column names preserved:", exported_df.columns.tolist() == df_clean.columns.tolist())

print("\nPreprocessing Summary:")
display(summary)


Export completed successfully!

Dataset file: preprocessed_output\preprocessed_articles_100k.csv
Summary report: preprocessed_output\preprocessing_summary.csv

Exported rows: 100,000
Exported columns: 9
File size: 50.41 MB

Export verification:
Row count preserved: True
Column names preserved: True

Preprocessing Summary:


,Metric,Value
0,Total Records,100000
1,Total Columns,9
2,Duplicate Rows,0
3,Duplicate Article IDs,0
4,Duplicate URLs,0
5,Missing Authors,58461
6,Missing Descriptions,123
7,Invalid Publication Timestamps,0
8,Invalid Scraping Timestamps,0
9,Missing Timezones,0
